In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_7_processed_dataset"
        / SAMPLE_ID
)

CELLS_DIR = PROCESSED_DIR / "cells"

In [ ]:
# ------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------

tracks = pd.read_csv(
    DATA_ROOT
    / "processed"
    / "stage_8_cell_tracking"
    / "tracks.csv"
)

print(f"Loaded {len(tracks):,} track records.")
print(f"Found {tracks['track_id'].nunique():,} tracks.")

tracks.head()

In [ ]:
cell_files = sorted(CELLS_DIR.glob("t*.csv"))

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")

## Stage 7A — Gap closing / track stitching

In [ ]:
# ============================================================
# Stage 7A — Gap closing / track stitching
# ============================================================
# Repairs "1 -> 0 -> 1" fragmentation: a track ends because a single
# frame's match was rejected (occlusion, missed segmentation, noise),
# and a "new" track starts a frame or two later that is really the
# same cell continuing. Stitches such fragments back into one
# track_id using position + volume similarity across the gap.

MAX_GAP = 2                 # max number of frames a track may vanish for
STITCH_MAX_DISTANCE = 15.0  # generous vs MAX_DISTANCE: spans >1 frame
STITCH_MAX_VOLUME_RATIO = 1.5


def stitch_tracks(tracks, max_gap=MAX_GAP,
                  max_distance=STITCH_MAX_DISTANCE,
                  max_volume_ratio=STITCH_MAX_VOLUME_RATIO):

    tracks = tracks.copy()
    last_frame = tracks["frame"].max()

    ends = (
        tracks.sort_values("frame")
        .groupby("track_id")
        .tail(1)
        .set_index("track_id")
    )
    starts = (
        tracks.sort_values("frame")
        .groupby("track_id")
        .head(1)
        .set_index("track_id")
    )

    # Candidate parents: tracks that end before the movie is over
    candidate_parents = ends[ends["frame"] < last_frame]
    # Candidate continuations: tracks that start after frame 0
    candidate_children = starts[starts["frame"] > 0]

    merges = {}  # child_track_id -> parent_track_id

    for parent_id, prow in candidate_parents.sort_values("frame").iterrows():

        window = candidate_children[
            (candidate_children["frame"] > prow["frame"]) &
            (candidate_children["frame"] <= prow["frame"] + max_gap) &
            (candidate_children.index != parent_id)
            ]
        if len(window) == 0:
            continue

        dist = np.sqrt(
            (window["z"] - prow["z"]) ** 2 +
            (window["y"] - prow["y"]) ** 2 +
            (window["x"] - prow["x"]) ** 2
        )
        vol_ratio = (
                np.maximum(window["volume"], prow["volume"])
                / np.minimum(window["volume"], prow["volume"])
        )

        ok = (dist <= max_distance) & (vol_ratio <= max_volume_ratio)
        candidates = dist[ok]
        if len(candidates) == 0:
            continue

        best_child = candidates.idxmin()

        if best_child in merges:
            continue  # fragment already claimed by an earlier-ending parent

        merges[best_child] = parent_id

    def resolve(track_id, _seen=None):
        _seen = _seen or set()
        while track_id in merges and track_id not in _seen:
            _seen.add(track_id)
            track_id = merges[track_id]
        return track_id

    tracks["track_id"] = tracks["track_id"].map(resolve)

    print(f"Stitched {len(merges)} fragmented track(s) "
          f"(gap <= {max_gap} frames).")

    return tracks


tracks = stitch_tracks(tracks)

## Stage 7B — Merge / split detection (segmentation artifacts)

In [ ]:
# ============================================================
# Stage 7B — Merge / split detection (segmentation artifacts)
# ============================================================
# Detects the "2 -> 1" signature of two touching cells getting
# segmented as one object: two tracks end in the same frame right
# next to a track whose volume looks like the sum of theirs.
#
# This does NOT auto-repair the tracks (splitting one track back into
# two mid-trajectory is ambiguous without more evidence). It flags
# candidates into segmentation_events for manual review and, more
# importantly, so Stage 8 can rule these out: a merge that later
# splits apart again looks like a division at the moment of
# separation, but should be excluded since it isn't one. A true
# division's daughter volumes are ~half of the *single* parent track,
# not equal to two previously-distinct tracks that briefly merged.

MERGE_MAX_DISTANCE = 15.0
MERGE_VOLUME_TOLERANCE = 0.25  # merged volume within 25% of vol_a + vol_b

segmentation_events = []

frames = sorted(tracks["frame"].unique())

for frame in frames[:-1]:

    this_frame = tracks[tracks["frame"] == frame]
    next_frame = tracks[tracks["frame"] == frame + 1]

    ending_here = this_frame[
        ~this_frame["track_id"].isin(next_frame["track_id"])
    ]

    if len(ending_here) < 2:
        continue

    for _, cand in next_frame.iterrows():

        dist = np.sqrt(
            (ending_here["z"] - cand["z"]) ** 2 +
            (ending_here["y"] - cand["y"]) ** 2 +
            (ending_here["x"] - cand["x"]) ** 2
        )
        nearby = ending_here[dist <= MERGE_MAX_DISTANCE]

        if len(nearby) < 2:
            continue

        for i in range(len(nearby)):
            for j in range(i + 1, len(nearby)):
                a, b = nearby.iloc[i], nearby.iloc[j]
                expected = a["volume"] + b["volume"]
                if abs(cand["volume"] - expected) / expected <= MERGE_VOLUME_TOLERANCE:
                    segmentation_events.append({
                        "type": "merge",
                        "frame": frame,
                        "track_a": a["track_id"],
                        "track_b": b["track_id"],
                        "merged_track": cand["track_id"],
                        "volume_a": a["volume"],
                        "volume_b": b["volume"],
                        "merged_volume": cand["volume"],
                    })

segmentation_events = pd.DataFrame(segmentation_events)
print(f"Flagged {len(segmentation_events)} candidate merge event(s).")
segmentation_events.head()

## Stage 7C — Track termination classification

For every track that ends, explain why.

In [ ]:
# ============================================================
# Stage 7C — Track termination classification
# ============================================================
# For every track that ends, explain why, checking these in priority
# order (first match wins):
#
#   reached_last_frame            movie simply ended
#   left_imaging_volume           last detection touches the FOV edge
#   merge_event                   flagged in Stage 7B segmentation_events
#   possible_division             2 daughters appear nearby, volumes ~sum
#   fragmented_likely_continuation a near-miss stitch (just outside
#                                  Stage 7A's stricter thresholds)
#   segmentation_failure          last-frame volume is an outlier vs.
#                                  the track's own history
#   unknown                       none of the above

FOV_EDGE_MARGIN = 5.0

DIVISION_MAX_GAP = 2
DIVISION_MAX_DISTANCE = 20.0
DIVISION_VOLUME_TOLERANCE = 0.3

FRAGMENT_MAX_GAP = 4
FRAGMENT_MAX_DISTANCE = 25.0
FRAGMENT_MAX_VOLUME_RATIO = 2.0

SEG_FAILURE_VOLUME_RATIO = 1.6  # last-frame volume vs. track's own median

# ---- Global imaging-volume bounds, estimated from all detections ----
all_detections = pd.concat(
    time_frames, keys=range(len(time_frames)), names=["frame"]
).reset_index(level=0)

FOV_BOUNDS = {
    "z_min": all_detections["z_min"].min(), "z_max": all_detections["z_max"].max(),
    "y_min": all_detections["y_min"].min(), "y_max": all_detections["y_max"].max(),
    "x_min": all_detections["x_min"].min(), "x_max": all_detections["x_max"].max(),
}

last_frame = tracks["frame"].max()

ends = (
    tracks.sort_values("frame")
    .groupby("track_id")
    .tail(1)
    .reset_index(drop=True)
)
starts = (
    tracks.sort_values("frame")
    .groupby("track_id")
    .head(1)
    .set_index("track_id")
)

# Track ids on either side of a flagged merge (Stage 7B)
merged_track_ids = set()
if len(segmentation_events):
    merged_track_ids |= set(segmentation_events["track_a"])
    merged_track_ids |= set(segmentation_events["track_b"])

records = []

for _, row in ends.iterrows():

    track_id = row["track_id"]
    frame = int(row["frame"])
    reason = None
    evidence = {}

    # ---- reached last frame ----
    if frame == last_frame:
        reason = "reached_last_frame"

    # ---- left imaging volume ----
    if reason is None:
        det_row = time_frames[frame].iloc[int(row["cell"])]
        touches_edge = (
                (det_row["z_min"] - FOV_BOUNDS["z_min"] <= FOV_EDGE_MARGIN) or
                (FOV_BOUNDS["z_max"] - det_row["z_max"] <= FOV_EDGE_MARGIN) or
                (det_row["y_min"] - FOV_BOUNDS["y_min"] <= FOV_EDGE_MARGIN) or
                (FOV_BOUNDS["y_max"] - det_row["y_max"] <= FOV_EDGE_MARGIN) or
                (det_row["x_min"] - FOV_BOUNDS["x_min"] <= FOV_EDGE_MARGIN) or
                (FOV_BOUNDS["x_max"] - det_row["x_max"] <= FOV_EDGE_MARGIN)
        )
        if touches_edge:
            reason = "left_imaging_volume"

    # ---- merge event ----
    if reason is None and track_id in merged_track_ids:
        reason = "merge_event"

    # ---- possible division: 2 daughters starting nearby, volumes ~sum ----
    if reason is None:
        window = starts[
            (starts["frame"] > frame) &
            (starts["frame"] <= frame + DIVISION_MAX_GAP) &
            (starts.index != track_id)
            ]
        if len(window) >= 2:
            dist = np.sqrt(
                (window["z"] - row["z"]) ** 2 +
                (window["y"] - row["y"]) ** 2 +
                (window["x"] - row["x"]) ** 2
            )
            nearby_ids = list(window.index[dist <= DIVISION_MAX_DISTANCE])
            for i in range(len(nearby_ids)):
                if reason is not None:
                    break
                for j in range(i + 1, len(nearby_ids)):
                    d1 = window.loc[nearby_ids[i]]
                    d2 = window.loc[nearby_ids[j]]
                    expected = row["volume"]
                    actual = d1["volume"] + d2["volume"]
                    if abs(actual - expected) / expected <= DIVISION_VOLUME_TOLERANCE:
                        reason = "possible_division"
                        evidence = {
                            "daughter_a": nearby_ids[i],
                            "daughter_b": nearby_ids[j],
                        }
                        break

    # ---- fragmented: looser near-miss stitch candidate ----
    if reason is None:
        window = starts[
            (starts["frame"] > frame) &
            (starts["frame"] <= frame + FRAGMENT_MAX_GAP) &
            (starts.index != track_id)
            ]
        if len(window):
            dist = np.sqrt(
                (window["z"] - row["z"]) ** 2 +
                (window["y"] - row["y"]) ** 2 +
                (window["x"] - row["x"]) ** 2
            )
            vol_ratio = (
                    np.maximum(window["volume"], row["volume"])
                    / np.minimum(window["volume"], row["volume"])
            )
            ok = (dist <= FRAGMENT_MAX_DISTANCE) & (vol_ratio <= FRAGMENT_MAX_VOLUME_RATIO)
            if ok.any():
                reason = "fragmented_likely_continuation"
                evidence = {"candidate_track": dist[ok].idxmin()}

    # ---- segmentation failure: last volume is an outlier vs. its history ----
    if reason is None:
        track_history = tracks[tracks["track_id"] == track_id]
        median_volume = track_history["volume"].median()
        if median_volume > 0:
            ratio = max(row["volume"], median_volume) / min(row["volume"], median_volume)
            if ratio >= SEG_FAILURE_VOLUME_RATIO:
                reason = "segmentation_failure"
                evidence = {"median_volume": median_volume, "last_volume": row["volume"]}

    # ---- fallback ----
    if reason is None:
        reason = "unknown"

    records.append({
        "track_id": track_id,
        "last_frame": frame,
        "reason": reason,
        **evidence,
    })

track_endings = pd.DataFrame(records)

# ------------------------------------------------------------
# Summary report
# ------------------------------------------------------------

REASON_ORDER = [
    "reached_last_frame",
    "left_imaging_volume",
    "fragmented_likely_continuation",
    "merge_event",
    "possible_division",
    "segmentation_failure",
    "unknown",
]

summary = track_endings["reason"].value_counts().reindex(REASON_ORDER, fill_value=0)

print(f"{len(track_endings)} total tracks")
for reason, count in summary.items():
    print(f"{count:4d} {reason.replace('_', ' ')}")

In [ ]:
# ------------------------------------------------------------
# Quick visual of the breakdown
# ------------------------------------------------------------

plt.figure(figsize=(8, 4))
summary.plot(kind="barh")
plt.xlabel("Number of tracks")
plt.title("Why each track ended")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

track_endings.head(20)

Save results

In [ ]:
output_dir = (
        DATA_ROOT
        / "processed"
        / "stage_9_track_stitching"
)

In [ ]:
import json

# ------------------------------------------------------------
# Save detections
# ------------------------------------------------------------

detections_df = pd.concat(
    time_frames,
    keys=range(len(time_frames)),
    names=["frame"],
).reset_index(level=0)

detections_df.to_csv(
    output_dir / "detections.csv",
    index=False,
)

# ------------------------------------------------------------
# Save tracks
# ------------------------------------------------------------

tracks.to_csv(
    output_dir / "tracks.csv",
    index=False,
)

# ------------------------------------------------------------
# Save segmentation events (candidate merges flagged in Stage 7B)
# ------------------------------------------------------------

segmentation_events.to_csv(
    output_dir / "segmentation_events.csv",
    index=False,
)

# ------------------------------------------------------------
# Save track termination classification
# ------------------------------------------------------------

track_endings.to_csv(
    output_dir / "track_endings.csv",
    index=False,
)

# ------------------------------------------------------------
# Save tracking parameters
# ------------------------------------------------------------

metadata = {
    "motion_compensation": True,
    "global_motion_estimator": "median (iteratively refined on matched pairs)",
    "gap_closing": True,
    "max_gap": MAX_GAP,
    "stitch_max_distance": STITCH_MAX_DISTANCE,
    "stitch_max_volume_ratio": STITCH_MAX_VOLUME_RATIO,
    "merge_detection": True,
    "merge_max_distance": MERGE_MAX_DISTANCE,
    "merge_volume_tolerance": MERGE_VOLUME_TOLERANCE,
}

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Saved Stage 9 results to: {output_dir}")